# H3b: Partial Equalization -- Do Top-k Singular Values Capture Muon's Advantage?

## Experiment Context and Motivation

This experiment is a direct follow-up to **H3** (the full polar factor test), which established that Muon's
polar factor -- projecting the gradient onto the closest orthogonal matrix via $G \to UV^T$ -- equalizes
**all** singular values to 1.0. This is qualitatively different from normalized SGD, which preserves the
*ratios* among singular values while only controlling their overall scale.

**The core question:** Does Muon *need* full equalization of the entire singular value spectrum, or does
equalizing only the top-k singular values capture most of the optimization advantage?

### Two Competing Hypotheses

| Hypothesis | Mechanism | Prediction |
|---|---|---|
| **(a) Dominant-SV suppression** | Muon works primarily by *clipping* the largest singular value, preventing any single gradient direction from dominating the update | k=1 should capture >50% of Muon's advantage |
| **(b) Full spectrum lifting** | Muon works by *amplifying* small singular values, ensuring the optimizer explores all directions equally | Full equalization (k=dim) is needed; small k captures little |

### The Partial Polar Factor Construction

Given a gradient matrix $G = U \Sigma V^T$, we construct a **partial polar factor** with parameter $k$:

$$\sigma_i^{\text{partial}} = \begin{cases} 1.0 & \text{if } i \leq k \text{ (top-k: equalized)} \\ \frac{\sigma_i}{\|\sigma_{k+1:}\|} \cdot \sqrt{d - k} & \text{if } i > k \text{ (remaining: normalized, structure preserved)} \end{cases}$$

The update direction is then $U \cdot \text{diag}(\sigma^{\text{partial}}) \cdot V^T$.

**Key design choice:** The remaining (non-equalized) singular values are *Frobenius-normalized* to have
total energy $\sqrt{d-k}$, matching the energy they would have if fully equalized. This isolates the
effect of *equalization* from the effect of *rescaling*.

### Experimental Protocol

- Sweep $k \in \{1, 2, 4, 8, 16, 32\}$ where $k=32$ recovers full Muon ($d = 32$)
- Compare against Normalized SGD (Frobenius-normalized momentum) as baseline
- Each method gets its own LR sweep for fair comparison
- 4-layer linear network, $32 \times 32$ weight matrices, 500 training steps, 10 random seeds

In [ ]:
"""
H3b: Partial Equalization -- Top-k SVs Capture Muon Advantage?
================================================================

MOTIVATION (from H3 surprise):
  Muon's polar factor equalizes ALL singular values to 1 (UV^T).
  This is qualitatively different from normalized SGD which preserves
  SV ratios. But does Muon NEED full equalization? Or does equalizing
  only the top-k singular values capture most of the advantage?

  This tests whether Muon's benefit comes from:
  (a) Suppressing the dominant SV (top-1 equalization suffices), or
  (b) Lifting ALL small SVs (full equalization needed).

PROTOCOL:
  Construct a "partial polar factor" optimizer:
    G = U diag(sigma) V^T
    For top-k SVs: set sigma_i = 1
    For remaining SVs: keep sigma_i / ||sigma|| (normalized but not equalized)
    Update = U diag(sigma_partial) V^T
  Sweep k in {1, 2, 4, 8, 16, 32 (=full Muon)}.
  Compare final loss for each k to full Muon and normalized SGD.

KEY TESTS:
  T1: Does k=1 (suppress only top SV) capture >50% of Muon's advantage?
  T2: Is there a sharp knee where most advantage is captured (k << dim)?
  T3: Does k=dim exactly recover Muon's performance (sanity check)?

Setup: 4-layer, 32x32, 500 steps, 10 seeds, LR swept per method.
"""

---
## 1. Imports and Environment Setup

In [ ]:
import numpy as np
import os

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

---
## 2. Experimental Hyperparameters

The network architecture and training configuration are deliberately kept small enough for exhaustive
sweeps across many seeds, while being deep enough (4 layers) that gradient conditioning matters.

**Why 32x32?** This gives 32 singular values per gradient matrix -- large enough to see spectral
structure, small enough for fast SVD at every step. The geometric progression of k-values
$\{1, 2, 4, 8, 16, 32\}$ lets us see whether the benefit of equalization saturates at a
fraction of the full dimension.

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 500
MOMENTUM = 0.9
NUM_SEEDS = 10
BATCH_SIZE = 64

In [ ]:
print("=" * 80)
print("EXPERIMENTAL CONFIGURATION")
print("=" * 80)
print(f"  Matrix dimension (DIM):      {DIM}")
print(f"  Number of layers:            {NUM_LAYERS}")
print(f"  Training steps:              {NUM_STEPS}")
print(f"  Momentum coefficient:        {MOMENTUM}")
print(f"  Number of random seeds:      {NUM_SEEDS}")
print(f"  Batch size:                  {BATCH_SIZE}")
print(f"  Total SVs per gradient:      {DIM} (= DIM)")
print(f"  Effective condition number")
print(f"    of init (I + 0.1*N(0,1)):  ~{1.0 + 0.1*np.sqrt(DIM):.1f} / {max(1.0 - 0.1*np.sqrt(DIM), 0.01):.2f} = ~{(1.0 + 0.1*np.sqrt(DIM))/max(1.0 - 0.1*np.sqrt(DIM), 0.01):.1f}")
print("=" * 80)

---
## 3. Sweep Configuration: k-values and Learning Rates

The k-values follow a geometric progression that doubles at each step, covering the full
range from "equalize only the dominant direction" ($k=1$) to "equalize everything" ($k=32 = d$, i.e., full Muon).

The learning rate grid is log-spaced from 0.001 to 0.05. Each method (each k-value and NormSGD)
gets its own optimal LR to ensure we are comparing *best achievable performance*, not sensitivity
to LR tuning.

In [ ]:
K_VALUES = [1, 2, 4, 8, 16, 32]  # 32 = full Muon (all SVs equalized)
LR_CANDIDATES = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.002, 0.001]

In [ ]:
print("Partial polar factor sweep points:")
for k in K_VALUES:
    frac = k / DIM * 100
    label = " (= full Muon, all SVs equalized)" if k == DIM else ""
    print(f"  k = {k:>3}  -->  {frac:5.1f}% of spectrum equalized{label}")
print(f"\nLR candidates ({len(LR_CANDIDATES)} values): {LR_CANDIDATES}")
print(f"LR range: [{min(LR_CANDIDATES)}, {max(LR_CANDIDATES)}]")

---
## 4. Core Algorithm: The Partial Polar Factor

This is the central construction of the experiment. Given a gradient matrix $G$, we compute its
thin SVD $G = U \Sigma V^T$ and then selectively equalize the singular values.

**Subtle design detail:** When $k < d$, the *remaining* singular values are not left at their
raw magnitudes. Instead, they are rescaled so that their total Frobenius energy equals $\sqrt{d - k}$
-- exactly what they would contribute if they were also equalized to 1.0. This means:

- **k=0** gives a direction-preserving, Frobenius-normalized gradient (like NormSGD but applied
  per-layer before momentum).
- **k=d** gives $U V^T$, the full polar factor (Muon).
- **Intermediate k** equalizes the top-k directions while preserving the *relative ratios* among
  the bottom $(d-k)$ directions, with matched total energy.

This energy matching is critical: without it, different k-values would have different effective
step sizes, confounding the comparison.

In [ ]:
def partial_polar(G, k):
    """
    Partial polar factor: equalize top-k SVs, normalize the rest.
    k=dim gives full polar factor (all SVs=1).
    k=0 gives Frobenius-normalized gradient.
    """
    U, sigma, Vt = np.linalg.svd(G, full_matrices=False)
    d = len(sigma)
    sigma_new = sigma.copy()
    norm = np.linalg.norm(sigma)
    if norm < 1e-15:
        return G

    # Top-k: set to 1
    sigma_new[:min(k, d)] = 1.0
    # Remaining: normalize to have consistent scale
    if k < d:
        remaining = sigma[k:]
        r_norm = np.linalg.norm(remaining)
        if r_norm > 1e-15:
            # Scale remaining so their total energy matches what equalized would give
            sigma_new[k:] = remaining / r_norm * np.sqrt(d - k)

    return U @ np.diag(sigma_new) @ Vt

In [ ]:
# ------------------------------------------------------------------
# Diagnostic: Demonstrate the partial polar factor on a sample gradient
# ------------------------------------------------------------------
print("=" * 80)
print("PARTIAL POLAR FACTOR -- DIAGNOSTIC DEMONSTRATION")
print("=" * 80)

rng_demo = np.random.RandomState(999)
G_demo = rng_demo.randn(DIM, DIM)
U_demo, sigma_demo, Vt_demo = np.linalg.svd(G_demo, full_matrices=False)

print(f"\nOriginal gradient SVs (first 10 of {DIM}):")
print(f"  {sigma_demo[:10].round(3)}")
print(f"  Condition number: {sigma_demo[0]/sigma_demo[-1]:.2f}")
print(f"  Frobenius norm:   {np.linalg.norm(sigma_demo):.4f}")

for k in [1, 4, 16, DIM]:
    pp = partial_polar(G_demo, k)
    _, sigma_pp, _ = np.linalg.svd(pp, full_matrices=False)
    print(f"\n  k={k:>2}: SVs = [{', '.join(f'{s:.3f}' for s in sigma_pp[:6])} ... {', '.join(f'{s:.3f}' for s in sigma_pp[-3:])}]")
    print(f"        Condition number: {sigma_pp[0]/sigma_pp[-1]:.4f}")
    print(f"        Frobenius norm:   {np.linalg.norm(sigma_pp):.4f}")
    n_equalized = np.sum(np.abs(sigma_pp - 1.0) < 1e-10)
    print(f"        SVs exactly = 1:  {n_equalized}")

print("\nKey observation: As k increases, more SVs are clamped to 1.0,")
print("reducing the condition number toward 1.0 (full isotropy at k=dim).")
print("The Frobenius norm is approximately preserved across all k values.")
print("=" * 80)

---
## 5. Network Architecture and Data Generation

We use a **deep linear network** -- a product of $L$ square matrices $W_L \cdots W_1$ -- as our
test bed. Despite having no nonlinearities, deep linear networks exhibit:

- **Gradient ill-conditioning** that worsens with depth (the gradient SVs span a wider range)
- **Saddle points** and non-trivial loss landscapes
- **Exact backpropagation** that lets us isolate optimizer effects from activation-function artifacts

Weight initialization: $W_i = I + 0.1 \cdot \mathcal{N}(0, 1)$, a near-identity perturbation
that ensures the initial forward pass is near-isometric while still breaking symmetry.

Data: Random Gaussian inputs and targets, scaled by 0.3 to keep initial losses moderate.

In [ ]:
def init_weights(seed):
    rng = np.random.RandomState(seed)
    return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(NUM_LAYERS)]

In [ ]:
def forward(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

In [ ]:
# ------------------------------------------------------------------
# Diagnostic: Inspect initial network conditions
# ------------------------------------------------------------------
print("=" * 80)
print("INITIAL NETWORK CONDITIONS (seed=42)")
print("=" * 80)

w_init = init_weights(42)
X_init, Y_init = make_data(42)

print(f"\nWeight matrices: {len(w_init)} layers, each {w_init[0].shape}")
for i, W in enumerate(w_init):
    sv = np.linalg.svd(W, compute_uv=False)
    print(f"  Layer {i}: SV range [{sv[-1]:.4f}, {sv[0]:.4f}], "
          f"cond = {sv[0]/sv[-1]:.2f}, "
          f"||W - I||_F = {np.linalg.norm(W - np.eye(DIM)):.4f}")

# Product of all layers (effective end-to-end transform)
W_prod = np.eye(DIM)
for W in w_init:
    W_prod = W @ W_prod
sv_prod = np.linalg.svd(W_prod, compute_uv=False)
print(f"\n  Product W_4...W_1: SV range [{sv_prod[-1]:.4f}, {sv_prod[0]:.4f}], "
      f"cond = {sv_prod[0]/sv_prod[-1]:.2f}")
print(f"  (Condition number grows exponentially with depth)")

# Initial loss and gradients
loss_init = compute_loss(w_init, X_init, Y_init)
grads_init = compute_gradients(w_init, X_init, Y_init)
print(f"\nInitial loss: {loss_init:.6f}")
print(f"\nInitial gradient singular value spectra:")
for i, g in enumerate(grads_init):
    sv_g = np.linalg.svd(g, compute_uv=False)
    print(f"  Layer {i} grad: SV range [{sv_g[-1]:.6f}, {sv_g[0]:.6f}], "
          f"cond = {sv_g[0]/sv_g[-1]:.1f}, "
          f"||G||_F = {np.linalg.norm(g):.6f}")

print("\nNote: Gradient condition numbers are typically MUCH larger than weight")
print("condition numbers, especially for deeper layers. This is precisely the")
print("ill-conditioning that Muon's polar factor addresses.")
print("=" * 80)

---
## 6. Training Loops

Two training functions implement the two optimizers under comparison:

1. **`train()`** -- Partial polar factor optimizer: computes $G \to \text{partial\_polar}(G, k)$
   before accumulating into momentum. When $k = d = 32$, this is exactly Muon.

2. **`train_norm_sgd()`** -- Normalized SGD baseline: accumulates raw gradients into momentum,
   then Frobenius-normalizes the momentum before taking the step. This preserves the singular
   value *ratios* in the update direction but controls overall magnitude.

**Important architectural detail:** In the partial polar version, the polar factor is applied
to the *raw gradient* before momentum accumulation. This means momentum sees a sequence of
(partially) orthogonalized directions. In NormSGD, momentum sees raw gradients and only the
final accumulated momentum is normalized. This matches the standard Muon implementation.

In [ ]:
def train(weights_init, X, Y, lr, k):
    """Train with partial polar factor (k=dim is full Muon, k=0 is normalized SGD)."""
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for i in range(NUM_LAYERS):
            pp = partial_polar(grads[i], k)
            mom[i] = MOMENTUM * mom[i] + pp
            weights[i] = weights[i] - lr * mom[i]
    return compute_loss(weights, X, Y)

In [ ]:
def train_norm_sgd(weights_init, X, Y, lr):
    """Normalized SGD (Frobenius normalization of momentum)."""
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for i in range(NUM_LAYERS):
            mom[i] = MOMENTUM * mom[i] + grads[i]
            v_norm = np.linalg.norm(mom[i], 'fro')
            step_dir = mom[i] / max(v_norm, 1e-12)
            weights[i] = weights[i] - lr * step_dir
    return compute_loss(weights, X, Y)

In [ ]:
def make_data(seed):
    rng = np.random.RandomState(seed)
    X = rng.randn(DIM, BATCH_SIZE) * 0.3
    Y = rng.randn(DIM, BATCH_SIZE) * 0.3
    return X, Y

---
## 7. Main Experiment: LR Sweep + Full Training + Hypothesis Testing

The experiment proceeds in two phases:

**Phase 1 -- Learning Rate Selection (3 seeds, quick sweep):**
For each k-value and for NormSGD, we sweep over 10 learning rate candidates and pick the one
that yields the lowest mean final loss over 3 seeds. This ensures each method is evaluated at
its best operating point, eliminating LR sensitivity as a confound.

**Phase 2 -- Full Evaluation (10 seeds, best LR):**
Using the selected LR, we train from scratch on all 10 seeds and report the mean final loss.
Statistical robustness comes from the seed diversity.

**Phase 3 -- Hypothesis Tests:**
- **T1:** Does k=1 capture >50% of the NormSGD-to-Muon gap? (Tests dominant-SV-suppression hypothesis)
- **T2:** Is there a sharp knee at k << dim? (Tests whether a small fraction of equalization suffices)
- **T3:** Does k=dim recover full Muon? (Sanity check on implementation correctness)

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H3b: PARTIAL EQUALIZATION -- Top-k SVs Capture Muon Advantage?")
    print("=" * 100)
    print(f"k values: {K_VALUES} (k={DIM} = full Muon)")
    print(f"Network: {NUM_LAYERS}-layer, {DIM}x{DIM}, {NUM_STEPS} steps, {NUM_SEEDS} seeds")
    print()

    # LR sweep for each k
    print("Phase 1: LR sweep per k...")
    print("-" * 80)
    print("  For each k, we train 3 seeds at each LR and pick the LR with lowest mean loss.")
    print("  This ensures fair comparison: each method gets its own optimal learning rate.")
    print("-" * 80)
    best_lrs = {}
    for k in K_VALUES:
        best_lr, best_loss = LR_CANDIDATES[-1], float('inf')
        all_lr_results = []
        for lr in LR_CANDIDATES:
            losses = []
            for s in seeds[:3]:
                X, Y = make_data(s)
                w = init_weights(s + 5000)
                fl = train(w, X, Y, lr, k)
                losses.append(fl)
            ml = np.mean([l for l in losses if np.isfinite(l)]) if any(np.isfinite(l) for l in losses) else float('inf')
            all_lr_results.append((lr, ml))
            if ml < best_loss:
                best_loss = ml
                best_lr = lr
        best_lrs[k] = best_lr
        print(f"  k={k:>3}: best_lr={best_lr:.4f}  (best_loss={best_loss:.6e})")
        # Show the full LR landscape for this k
        print(f"         LR landscape: ", end="")
        for lr, ml in all_lr_results:
            marker = " <--" if lr == best_lr else ""
            if np.isfinite(ml):
                print(f"{lr:.3f}:{ml:.2e}{marker}  ", end="")
            else:
                print(f"{lr:.3f}:diverged  ", end="")
        print()

    # Also sweep for normalized SGD
    print(f"\n  Sweeping LR for Normalized SGD baseline...")
    best_norm_lr, best_norm_loss = LR_CANDIDATES[-1], float('inf')
    for lr in LR_CANDIDATES:
        losses = []
        for s in seeds[:3]:
            X, Y = make_data(s)
            w = init_weights(s + 5000)
            fl = train_norm_sgd(w, X, Y, lr)
            losses.append(fl)
        ml = np.mean([l for l in losses if np.isfinite(l)]) if any(np.isfinite(l) for l in losses) else float('inf')
        if ml < best_norm_loss:
            best_norm_loss = ml
            best_norm_lr = lr
    print(f"  NormSGD: best_lr={best_norm_lr:.4f}  (best_loss={best_norm_loss:.6e})")

    print(f"\n  Summary of selected learning rates:")
    print(f"  {'Method':>10} | {'Best LR':>8}")
    print(f"  {'-'*10}-+-{'-'*8}")
    for k in K_VALUES:
        label = f"k={k}" + (" (Muon)" if k == DIM else "")
        print(f"  {label:>10} | {best_lrs[k]:>8.4f}")
    print(f"  {'NormSGD':>10} | {best_norm_lr:>8.4f}")

    # Full training
    print(f"\n{'='*80}")
    print("Phase 2: Full training with selected LRs across all 10 seeds...")
    print(f"{'='*80}")
    results = {}
    for k in K_VALUES:
        losses = []
        for s in seeds:
            X, Y = make_data(s)
            w = init_weights(s + 5000)
            fl = train(w, X, Y, best_lrs[k], k)
            losses.append(fl)
        finite = [l for l in losses if np.isfinite(l)]
        results[k] = np.mean(finite) if finite else float('inf')
        std_val = np.std(finite) if len(finite) > 1 else 0.0
        print(f"  k={k:>3}: mean_loss = {results[k]:.6e}  (std = {std_val:.2e}, {len(finite)}/{len(losses)} finite)")

    norm_losses = []
    for s in seeds:
        X, Y = make_data(s)
        w = init_weights(s + 5000)
        fl = train_norm_sgd(w, X, Y, best_norm_lr)
        norm_losses.append(fl)
    norm_finite = [l for l in norm_losses if np.isfinite(l)]
    norm_result = np.mean(norm_finite)
    norm_std = np.std(norm_finite) if len(norm_finite) > 1 else 0.0
    print(f"  NormSGD: mean_loss = {norm_result:.6e}  (std = {norm_std:.2e}, {len(norm_finite)}/{len(norm_losses)} finite)")

    # Results
    print(f"\n{'=' * 100}")
    print("RESULTS: Final Loss vs Number of Equalized SVs")
    print(f"{'=' * 100}")

    muon_loss = results[DIM]
    norm_loss = norm_result

    print(f"\n  {'k':>5}  {'Loss':>14}  {'vs NormSGD':>12}  {'vs Muon':>10}  {'% of gap':>10}")
    print("  " + "-" * 55)

    total_gap = norm_loss - muon_loss if np.isfinite(norm_loss) and np.isfinite(muon_loss) else 1.0

    print(f"  {'NormSGD':>5}  {norm_loss:>14.6e}  {'ref':>12}  {norm_loss/max(muon_loss,1e-30):>10.1f}x  {'0%':>10}")
    for k in K_VALUES:
        r = results[k]
        vs_norm = norm_loss / max(r, 1e-30)
        vs_muon = r / max(muon_loss, 1e-30)
        gap_captured = (norm_loss - r) / max(total_gap, 1e-30) * 100 if total_gap > 1e-30 else 0
        marker = " <-- full Muon" if k == DIM else ""
        print(f"  {k:>5}  {r:>14.6e}  {vs_norm:>12.1f}x  {vs_muon:>10.2f}x  {gap_captured:>9.1f}%{marker}")

    # Print interpretation of the monotonic trend
    print(f"\n  Interpretation:")
    print(f"    NormSGD loss:  {norm_loss:.6e}")
    print(f"    Full Muon loss (k={DIM}): {muon_loss:.6e}")
    print(f"    Total gap:     {total_gap:.6e}")
    print(f"    This gap represents the full optimization advantage of Muon over NormSGD.")

    # Hypothesis tests
    print(f"\n\n{'=' * 100}")
    print("HYPOTHESIS TESTS")
    print(f"{'=' * 100}")

    gap_k1 = (norm_loss - results[1]) / max(total_gap, 1e-30) * 100
    t1 = gap_k1 > 50
    print(f"\n  T1: k=1 captures >50% of Muon advantage?")
    print(f"      Gap captured: {gap_k1:.1f}%")
    print(f"      Loss at k=1: {results[1]:.6e}")
    print(f"      Interpretation: Equalizing ONLY the dominant singular value {'IS' if t1 else 'is NOT'}")
    print(f"      sufficient to capture the majority of Muon's advantage over NormSGD.")
    if t1:
        print(f"      This supports hypothesis (a): the primary mechanism is suppressing the")
        print(f"      dominant gradient direction that would otherwise monopolize the update step.")
    else:
        print(f"      This supports hypothesis (b): Muon's advantage requires lifting MANY small")
        print(f"      singular values, not just suppressing the dominant one.")
    print(f"      --> {'PASS' if t1 else 'FAIL'}")

    # Find knee
    gaps = [(k, (norm_loss - results[k]) / max(total_gap, 1e-30) * 100) for k in K_VALUES]
    knee_k = None
    for k, pct in gaps:
        if pct > 80:
            knee_k = k
            break
    t2 = knee_k is not None and knee_k < DIM // 2
    print(f"\n  T2: Sharp knee at k << dim (>80% gap captured at k < {DIM//2})?")
    print(f"      Progressive gap capture:")
    for k, pct in gaps:
        bar = "#" * int(pct / 2)
        print(f"        k={k:>3}: {pct:6.1f}% |{bar}")
    if knee_k:
        print(f"      Knee at k={knee_k} ({gaps[K_VALUES.index(knee_k)][1]:.1f}% gap)")
    else:
        print(f"      No knee found (gap never reaches 80% before k=dim)")
    print(f"      --> {'PASS' if t2 else 'FAIL'}")

    t3 = abs(results[DIM] - muon_loss) / max(muon_loss, 1e-30) < 0.05
    print(f"\n  T3: k=dim recovers full Muon within 5%?")
    print(f"      k={DIM} loss={results[DIM]:.6e}, Muon loss={muon_loss:.6e}")
    print(f"      Relative difference: {abs(results[DIM] - muon_loss) / max(muon_loss, 1e-30) * 100:.4f}%")
    print(f"      (This is a sanity check: k=dim should exactly recover Muon by construction,")
    print(f"       since setting ALL singular values to 1.0 gives the polar factor UV^T.)")
    print(f"      --> {'PASS (sanity check)' if t3 else 'FAIL (implementation bug?)'}")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

In [ ]:
if __name__ == '__main__':
    main()

---
## 8. Interpretation and Conclusions

### What This Experiment Reveals

The partial polar factor construction creates a clean interpolation between two extremes:
- **k=0 (or NormSGD):** Preserves all singular value ratios; only controls overall magnitude
- **k=d (full Muon):** Destroys all singular value information; every direction gets equal weight

By sweeping k, we can determine the *minimal intervention* that captures Muon's benefit.

### Implications for the RG Gauge-Fixing Hypothesis

If Muon's advantage is captured at small k, this suggests that the *gauge-fixing* interpretation
of Muon operates primarily on the **low-rank subspace** of the gradient -- the dominant few
directions that carry most of the optimization signal. The remaining directions are "noise" that
doesn't need equalization.

If full equalization is needed, this suggests that Muon's gauge-fixing operates on the *entire
tangent space* of the weight manifold, and that even small singular values carry meaningful
optimization information that benefits from amplification.

### Connection to H3 (Full Polar Factor)

H3 established that the polar factor (UV^T) is the key mechanism. H3b refines this by asking
*which part* of the polar factor matters most. Together, these experiments map out the
"equalization landscape" of Muon's singular value manipulation.

### What To Look For

- **Monotonic decrease in loss as k increases:** Expected, since more equalization brings us
  closer to full Muon.
- **Diminishing returns at large k:** If the curve flattens early, most benefit comes from
  equalizing a few top SVs.
- **Linear scaling with k:** Would suggest each equalized SV contributes equally -- no special
  role for top vs. bottom of the spectrum.
- **Threshold behavior:** A sharp jump at some critical k would suggest a phase transition in
  optimization quality.